# LeetCode 2848 — Points That Intersect With Cars bằng Python

Link đề: https://leetcode.com/problems/points-that-intersect-with-cars/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Prefix Sum, Intervals

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def numberOfPoints(self, nums: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi chiếc xe chiếm một đoạn `[start, end]` trên trục số (gồm cả hai đầu); đếm số **điểm nguyên** được **ít nhất một xe** phủ tới.
2. Biết **tự tính tay**: vẽ các đoạn lên trục số, gạch các điểm được phủ, đếm không trùng lặp.
3. Học các công cụ Python: `range(start, end + 1)`, `set` và `len(set)`, danh sách đánh dấu (mảng boolean), `sorted` với `key`, **mảng hiệu (difference array)** và cộng dồn.
4. Đi từ **Brute Force: bỏ mọi điểm vào `set`** → **mảng đánh dấu `covered[101]`** → **mảng hiệu + quét (sweep)** → **sắp xếp và gộp đoạn (merge intervals)**.
5. Chứng minh vì sao mảng hiệu đếm đúng, và tránh lỗi kinh điển: dùng `range(start, end)` (mất điểm cuối), cộng độ dài các đoạn mà quên trùng lặp, gộp đoạn sai khi hai đoạn chỉ chạm nhau.

## 2. Đề bài & ví dụ

Bạn được cho một mảng 2 chiều `nums` biểu diễn toạ độ của các **xe ô tô** đang đậu trên một **trục số**. Với mỗi `i`, `nums[i] = [start_i, end_i]` nghĩa là chiếc xe thứ `i` bắt đầu ở điểm `start_i` và kết thúc ở điểm `end_i` (xe phủ **mọi điểm nguyên** từ `start_i` đến `end_i`, **bao gồm cả hai đầu**).

Trả về **số lượng điểm nguyên** trên trục số **bị phủ bởi ít nhất một chiếc xe**.

| Ví dụ | `nums` | Output | Giải thích |
|---|---|---|---|
| 1 | `[[3,6],[1,5],[4,7]]` | `7` | Mọi điểm nguyên từ `1` đến `7` đều bị phủ: `1, 2, 3, 4, 5, 6, 7` ⇒ 7 điểm. |
| 2 | `[[1,3],[5,8]]` | `7` | Các điểm bị phủ: `1, 2, 3, 5, 6, 7, 8` ⇒ 7 điểm (điểm `4` không có xe nào). |

Constraints:

- `1 <= nums.length <= 100`
- `nums[i].length == 2`
- `1 <= start_i <= end_i <= 100`

In [1]:
# Liệt kê tay các điểm của ví dụ 2
nums = [[1, 3], [5, 8]]
diem = set()
for s, e in nums:
    for p in range(s, e + 1):
        diem.add(p)
print(sorted(diem), len(diem))     # [1, 2, 3, 5, 6, 7, 8] 7

[1, 2, 3, 5, 6, 7, 8] 7


## 3. Kiến thức nền

### 3.1. `range(start, end + 1)` — đoạn đóng hai đầu

`range(a, b)` đi từ `a` tới `b − 1` (không gồm `b`). Vì đề nói xe phủ **cả** điểm `end`, ta phải dùng `range(start, end + 1)`.

In [2]:
print(list(range(3, 6)))        # [3, 4, 5]    <- thiếu 6
print(list(range(3, 6 + 1)))    # [3, 4, 5, 6] <- đủ cả hai đầu

[3, 4, 5]
[3, 4, 5, 6]


### 3.2. `set` — tập hợp không trùng lặp

`set` tự bỏ phần tử trùng. `s.add(x)` thêm một phần tử; `s.update(iterable)` thêm nhiều phần tử; `len(s)` là số phần tử khác nhau. Rất hợp để "đếm số điểm khác nhau".

In [3]:
s = set()
s.add(3)
s.update(range(3, 6))     # 3, 4, 5 (3 đã có)
print(s, len(s))          # {3, 4, 5} 3

{3, 4, 5} 3


### 3.3. Danh sách đánh dấu (mảng boolean)

Vì toạ độ nằm trong `1..100`, ta có thể dùng list `covered = [False] * 101` để đánh dấu `covered[p] = True` khi điểm `p` được phủ. Đếm bằng `sum(covered)` (True tính là 1).

In [4]:
covered = [False] * 8
for p in (2, 3, 4):
    covered[p] = True
print(covered)            # [False, False, True, True, True, False, False, False]
print(sum(covered))       # 3

[False, False, True, True, True, False, False, False]
3


### 3.4. Gán lát cắt: `a[l:r] = [...]`

Có thể đánh dấu cả đoạn một lúc bằng cắt lát: `covered[s:e + 1] = [True] * (e - s + 1)`. Nhanh hơn vòng `for` nhỏ và gọn.

In [5]:
covered = [False] * 8
covered[2:5] = [True] * 3          # đánh dấu 2, 3, 4
print(covered)

[False, False, True, True, True, False, False, False]


### 3.5. Mảng hiệu (difference array) và cộng dồn

Muốn "cộng 1 cho mọi vị trí trong đoạn `[s, e]`" mà không duyệt từng vị trí: đặt `diff[s] += 1` và `diff[e + 1] -= 1`. Sau đó **cộng dồn** `cur += diff[i]` khi quét `i` từ trái sang phải: `cur` chính là **số đoạn đang phủ vị trí `i`**.

In [6]:
diff = [0] * 10
for s, e in [(2, 4), (3, 6)]:
    diff[s] += 1
    diff[e + 1] -= 1
print(diff)
cur = 0
for i in range(10):
    cur += diff[i]
    print(i, cur, end=" | ")

[0, 0, 1, 1, 0, -1, 0, -1, 0, 0]
0 0 | 1 0 | 2 1 | 3 2 | 4 2 | 5 1 | 6 1 | 7 0 | 8 0 | 9 0 | 

### 3.6. Sắp xếp theo khoá: `sorted(nums, key=...)`

`sorted(nums)` với `nums` là list các list sẽ so sánh theo phần tử đầu `start`, rồi `end`. Ta dùng để **sắp các đoạn theo điểm đầu** trước khi gộp.

In [7]:
nums = [[3, 6], [1, 5], [4, 7]]
print(sorted(nums))                          # [[1, 5], [3, 6], [4, 7]]
print(sorted(nums, key=lambda x: x[0]))      # cùng kết quả

[[1, 5], [3, 6], [4, 7]]
[[1, 5], [3, 6], [4, 7]]


### 3.7. Phép `max` để cập nhật điểm cuối khi gộp đoạn

Gộp hai đoạn chồng nhau `[1, 5]` và `[3, 6]` được `[1, max(5, 6)] = [1, 6]`. Điểm cuối mới là `max` của hai điểm cuối (vì đoạn thứ hai có thể nằm gọn bên trong đoạn thứ nhất).

In [8]:
print(max(5, 6))      # 6  -> [1,5] + [3,6] = [1,6]
print(max(7, 4))      # 7  -> [1,7] + [3,4] = [1,7]  (đoạn sau nằm gọn bên trong)

6
7


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Vẽ ví dụ 1 lên trục số: `[3,6]`, `[1,5]`, `[4,7]`.

```
điểm:   1  2  3  4  5  6  7
[3,6]:        ■  ■  ■  ■
[1,5]:  ■  ■  ■  ■  ■
[4,7]:           ■  ■  ■  ■
phủ:    ✔  ✔  ✔  ✔  ✔  ✔  ✔     -> 7 điểm
```

Nếu cộng độ dài từng đoạn (`4 + 5 + 4 = 13`) thì **sai** vì các điểm `3, 4, 5, 6` bị đếm nhiều lần. Ta cần đếm mỗi điểm **một lần**.

Ba cách nghĩ tự nhiên:

1. **Tập hợp:** bỏ mọi điểm của mọi đoạn vào một `set`, `len(set)` là đáp án (set tự loại trùng).
2. **Đánh dấu:** có tối đa 100 điểm; tạo bảng 100 ô, ô nào được xe phủ thì tô; đếm số ô tô.
3. **Gộp đoạn:** sắp đoạn theo điểm đầu, gộp những đoạn chồng nhau thành đoạn lớn rồi cộng độ dài (không còn trùng lặp).

In [9]:
nums = [[3, 6], [1, 5], [4, 7]]
tong_do_dai = sum(e - s + 1 for s, e in nums)
print("cộng độ dài từng đoạn (sai vì trùng):", tong_do_dai)       # 13
print("số điểm khác nhau (đúng):", len({p for s, e in nums for p in range(s, e + 1)}))   # 7

cộng độ dài từng đoạn (sai vì trùng): 13
số điểm khác nhau (đúng): 7


## 5. Cách 1 — Brute Force / baseline: bỏ mọi điểm vào `set`

**Ý tưởng:** với mỗi đoạn `[s, e]`, thêm các điểm `s, s+1, ..., e` vào một `set`; cuối cùng trả về `len(set)`.

Bản dịch thẳng của đề, rất dễ đúng, dùng làm **oracle**.

In [10]:
from typing import List

def points_set(nums: List[List[int]]) -> int:
    seen = set()
    for s, e in nums:
        for p in range(s, e + 1):      # gồm cả điểm cuối e
            seen.add(p)
    return len(seen)

In [11]:
print(points_set([[3, 6], [1, 5], [4, 7]]))     # 7
print(points_set([[1, 3], [5, 8]]))              # 7
print(points_set([[5, 5]]))                      # 1 (xe chiếm đúng một điểm)

7
7
1


**Độ phức tạp:** gọi `n = len(nums)`, `M` là toạ độ lớn nhất (`<= 100`). Mỗi đoạn có tối đa `M` điểm ⇒ Time `O(n·M)`; Space `O(M)` cho `set` (tối đa `M` điểm khác nhau).

**Nút thắt:** nếu toạ độ lớn tới `10^9` (không còn vừa trong bảng 100 ô), việc liệt kê từng điểm là không khả thi. Ta cần cách chỉ phụ thuộc số đoạn `n`, không phụ thuộc độ dài đoạn.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — đánh dấu thay cho `set`.** Toạ độ trong `1..100`, nên một list 101 ô thay `set`, nhanh và đơn giản: `covered[p] = True`.

**Đột phá 2 — mảng hiệu.** Thay vì đánh dấu từng điểm của đoạn, chỉ cập nhật **hai chỗ**: `diff[s] += 1` (bắt đầu phủ) và `diff[e + 1] -= 1` (hết phủ). Quét cộng dồn: `cur` = số xe phủ điểm `i`; điểm `i` được phủ khi `cur > 0`. Mỗi đoạn tốn `O(1)` thay vì `O(độ dài)`.

**Đột phá 3 — gộp đoạn.** Sau khi sắp theo điểm đầu, duyệt một lần và giữ "đoạn hiện tại" `[cs, ce]`:

- Nếu đoạn kế `[s, e]` có `s <= ce` (chồng hoặc chạm điểm): mở rộng `ce = max(ce, e)`.
- Ngược lại: chốt đoạn hiện tại (cộng `ce − cs + 1`), bắt đầu đoạn mới.

Đây là mẫu **merge intervals** kinh điển, độ phức tạp chỉ phụ thuộc `n`, không phụ thuộc toạ độ.

In [12]:
nums = [[3, 6], [1, 5], [4, 7]]
diff = [0] * 102
for s, e in nums:
    diff[s] += 1
    diff[e + 1] -= 1
cur = 0
count = 0
for i in range(1, 101):
    cur += diff[i]
    if cur > 0:
        count += 1
print("số điểm được phủ:", count)      # 7

số điểm được phủ: 7


## 7. Các cách tiếp theo

Ta giảng thêm 3 cách nữa.

### Cách 2 — Mảng đánh dấu `covered[101]`

**Từng đoạn:**

- `covered = [False] * 102` (chỉ số `0..101`, ta dùng `1..100`).
- Với mỗi đoạn `[s, e]`, gán `covered[s:e + 1] = [True] * (e − s + 1)`.
- Đáp án là `sum(covered)`.

In [13]:
def points_marks(nums: List[List[int]]) -> int:
    covered = [False] * 102
    for s, e in nums:
        covered[s:e + 1] = [True] * (e - s + 1)
    return sum(covered)

In [14]:
print(points_marks([[3, 6], [1, 5], [4, 7]]))     # 7
print(points_marks([[1, 3], [5, 8]]))              # 7

7
7


**Code nộp LeetCode (Cách 2):**

```python
from typing import List

class Solution:
    def numberOfPoints(self, nums: List[List[int]]) -> int:
        covered = [False] * 102
        for s, e in nums:
            covered[s:e + 1] = [True] * (e - s + 1)
        return sum(covered)
```

**Độ phức tạp:** Time `O(n·M)`, Space `O(M)` với `M = 100`.

### Cách 3 — Mảng hiệu + quét cộng dồn

**Từng đoạn:**

- `diff = [0] * 102`. Mỗi đoạn `[s, e]`: `diff[s] += 1`, `diff[e + 1] -= 1` (chỉ số `e + 1 ≤ 101` nên mảng cần 102 ô).
- Quét `i` từ `1` đến `100`: `cur += diff[i]`; nếu `cur > 0` thì `count += 1`.

In [15]:
def points_diff(nums: List[List[int]]) -> int:
    diff = [0] * 102
    for s, e in nums:
        diff[s] += 1
        diff[e + 1] -= 1
    cur = count = 0
    for i in range(1, 101):
        cur += diff[i]
        if cur > 0:
            count += 1
    return count

In [16]:
print(points_diff([[3, 6], [1, 5], [4, 7]]))     # 7
print(points_diff([[1, 3], [5, 8]]))              # 7

7
7


**Code nộp LeetCode (Cách 3):**

```python
from typing import List

class Solution:
    def numberOfPoints(self, nums: List[List[int]]) -> int:
        diff = [0] * 102
        for s, e in nums:
            diff[s] += 1
            diff[e + 1] -= 1
        cur = count = 0
        for i in range(1, 101):
            cur += diff[i]
            if cur > 0:
                count += 1
        return count
```

**Độ phức tạp:** Time `O(n + M)` (mỗi đoạn `O(1)`, rồi quét `M`), Space `O(M)`.

### Cách 4 — Sắp xếp và gộp đoạn (merge intervals)

**Từng đoạn:**

- Sắp `nums` theo điểm đầu.
- Giữ `[cs, ce]` là đoạn gộp hiện tại (lúc đầu là đoạn đầu tiên).
- Với mỗi đoạn `[s, e]` tiếp theo: nếu `s <= ce` thì `ce = max(ce, e)`; ngược lại `total += ce − cs + 1` rồi `cs, ce = s, e`.
- Cuối cùng cộng đoạn đang giữ.

In [17]:
def points_merge(nums: List[List[int]]) -> int:
    nums = sorted(nums)
    cs, ce = nums[0]
    total = 0
    for s, e in nums[1:]:
        if s <= ce:                  # chồng hoặc chạm điểm
            ce = max(ce, e)
        else:                        # đoạn mới tách rời
            total += ce - cs + 1
            cs, ce = s, e
    total += ce - cs + 1
    return total

In [18]:
print(points_merge([[3, 6], [1, 5], [4, 7]]))     # 7
print(points_merge([[1, 3], [5, 8]]))              # 7
print(points_merge([[1, 3], [4, 6]]))              # 6 (hai đoạn kề nhau nhưng không chồng)

7
7
6


**Code nộp LeetCode (Cách 4):**

```python
from typing import List

class Solution:
    def numberOfPoints(self, nums: List[List[int]]) -> int:
        nums = sorted(nums)
        cs, ce = nums[0]
        total = 0
        for s, e in nums[1:]:
            if s <= ce:
                ce = max(ce, e)
            else:
                total += ce - cs + 1
                cs, ce = s, e
        return total + ce - cs + 1
```

**Độ phức tạp:** sắp xếp `O(n log n)`, quét `O(n)` ⇒ Time `O(n log n)`, Space `O(1)` phụ (hoặc `O(n)` nếu `sorted` sao chép). Điểm mạnh: **không phụ thuộc toạ độ**, dùng được khi toạ độ rất lớn.

**Lưu ý** hai đoạn `[1, 3]` và `[4, 6]` **kề nhau nhưng không chồng** (không có điểm nguyên chung): điều kiện gộp là `s <= ce` (không phải `s <= ce + 1`); khi đó mỗi đoạn tính độ dài riêng, tổng vẫn `3 + 3 = 6` đúng.

## 8. TRACE từng bước

Trace Cách 4 (gộp đoạn) trên `nums = [[3,6],[1,5],[4,7]]`. Sau khi sắp xếp: `[[1,5],[3,6],[4,7]]`.

| bước | đoạn mới `[s, e]` | đoạn đang giữ `[cs, ce]` | `s <= ce`? | hành động | `total` |
|---|---|---|---|---|---|
| khởi tạo | — | `[1, 5]` | — | lấy đoạn đầu | 0 |
| 1 | `[3, 6]` | `[1, 5]` | `3 <= 5` có | `ce = max(5, 6) = 6` | 0 |
| 2 | `[4, 7]` | `[1, 6]` | `4 <= 6` có | `ce = max(6, 7) = 7` | 0 |
| kết thúc | — | `[1, 7]` | — | cộng `7 − 1 + 1` | **7** |

In [19]:
def trace_merge(nums):
    nums = sorted(nums)
    print("sau khi sắp xếp:", nums)
    cs, ce = nums[0]
    total = 0
    print(f"khởi tạo: đoạn giữ = [{cs}, {ce}]")
    for s, e in nums[1:]:
        if s <= ce:
            ce_moi = max(ce, e)
            print(f"đoạn [{s}, {e}]: {s} <= {ce} -> gộp, ce = max({ce}, {e}) = {ce_moi}")
            ce = ce_moi
        else:
            total += ce - cs + 1
            print(f"đoạn [{s}, {e}]: {s} > {ce} -> chốt [{cs}, {ce}] (+{ce - cs + 1}), bắt đầu đoạn mới")
            cs, ce = s, e
    total += ce - cs + 1
    print(f"kết thúc: chốt [{cs}, {ce}] (+{ce - cs + 1}) -> tổng = {total}")
    return total

trace_merge([[3, 6], [1, 5], [4, 7]])
print("---")
trace_merge([[1, 3], [5, 8], [2, 4], [10, 10]])

sau khi sắp xếp: [[1, 5], [3, 6], [4, 7]]
khởi tạo: đoạn giữ = [1, 5]
đoạn [3, 6]: 3 <= 5 -> gộp, ce = max(5, 6) = 6
đoạn [4, 7]: 4 <= 6 -> gộp, ce = max(6, 7) = 7
kết thúc: chốt [1, 7] (+7) -> tổng = 7
---
sau khi sắp xếp: [[1, 3], [2, 4], [5, 8], [10, 10]]
khởi tạo: đoạn giữ = [1, 3]
đoạn [2, 4]: 2 <= 3 -> gộp, ce = max(3, 4) = 4
đoạn [5, 8]: 5 > 4 -> chốt [1, 4] (+4), bắt đầu đoạn mới
đoạn [10, 10]: 10 > 8 -> chốt [5, 8] (+4), bắt đầu đoạn mới
kết thúc: chốt [10, 10] (+1) -> tổng = 9


9

In [20]:
# Trace mảng hiệu: ví dụ 2 [[1,3],[5,8]]
def trace_diff(nums, upto=10):
    diff = [0] * 102
    for s, e in nums:
        diff[s] += 1
        diff[e + 1] -= 1
    print("diff[1..%d] =" % upto, diff[1:upto + 1])
    cur = count = 0
    for i in range(1, upto + 1):
        cur += diff[i]
        if cur > 0:
            count += 1
        print(f"i={i:2}: diff={diff[i]:2}, cur={cur}, {'được phủ' if cur > 0 else 'không phủ':10} | count={count}")
    return count

trace_diff([[1, 3], [5, 8]])

diff[1..10] = [1, 0, 0, -1, 1, 0, 0, 0, -1, 0]
i= 1: diff= 1, cur=1, được phủ   | count=1
i= 2: diff= 0, cur=1, được phủ   | count=2
i= 3: diff= 0, cur=1, được phủ   | count=3
i= 4: diff=-1, cur=0, không phủ  | count=3
i= 5: diff= 1, cur=1, được phủ   | count=4
i= 6: diff= 0, cur=1, được phủ   | count=5
i= 7: diff= 0, cur=1, được phủ   | count=6
i= 8: diff= 0, cur=1, được phủ   | count=7
i= 9: diff=-1, cur=0, không phủ  | count=7
i=10: diff= 0, cur=0, không phủ  | count=7


7

## 9. Code nộp LeetCode

Dùng Cách 3 (mảng hiệu + quét): đơn giản, không phụ thuộc thứ tự đoạn, và đúng với ràng buộc `1..100`.

In [21]:
from typing import List

class Solution:
    def numberOfPoints(self, nums: List[List[int]]) -> int:
        diff = [0] * 102                 # diff[e + 1] với e <= 100 nên cần chỉ số 101
        for s, e in nums:
            diff[s] += 1                 # bắt đầu được phủ tại s
            diff[e + 1] -= 1             # hết phủ sau e
        cur = count = 0
        for i in range(1, 101):
            cur += diff[i]               # cur = số xe phủ điểm i
            if cur > 0:
                count += 1
        return count

In [22]:
print(Solution().numberOfPoints([[3, 6], [1, 5], [4, 7]]))     # 7
print(Solution().numberOfPoints([[1, 3], [5, 8]]))              # 7

7
7


## 10. Vì sao thuật toán đúng

**Mảng hiệu.** Định nghĩa `cover(i)` là số xe phủ điểm `i`, tức số `j` thoả `s_j <= i <= e_j`.

Ta chứng minh sau khi quét tới `i`, `cur = cover(i)`. Ta có `cur_i = Σ_{k <= i} diff[k]`. Mỗi xe `j` đóng góp `+1` tại `k = s_j` (nếu `s_j <= i`) và `−1` tại `k = e_j + 1` (nếu `e_j + 1 <= i`). Đóng góp tổng của xe `j` vào `cur_i`:

- `i < s_j`: không đóng góp gì ⇒ `0`.
- `s_j <= i <= e_j`: chỉ có `+1` ⇒ `1`.
- `i >= e_j + 1`: `+1` và `−1` ⇒ `0`.

Vậy xe `j` đóng góp đúng `1` khi và chỉ khi `s_j <= i <= e_j`, tức khi nó phủ điểm `i`. Cộng theo mọi xe: `cur_i = cover(i)`.

**Đếm điểm.** Điểm `i` bị phủ bởi ít nhất một xe ⇔ `cover(i) > 0` ⇔ `cur_i > 0`. Ta đếm các `i ∈ [1, 100]` thoả điều đó: chính là đáp án.

**Chỉ số `e + 1 = 101`.** Vì `e <= 100`, chỉ số lớn nhất ta ghi là `101`; mảng 102 ô (chỉ số `0..101`) là đủ.

**Gộp đoạn (Cách 4) cũng đúng.** Sau khi sắp theo điểm đầu, nếu `s <= ce` thì đoạn mới có điểm chung với đoạn gộp (vì mọi điểm từ `s` đến `ce` đã nằm trong cả hai), hợp của chúng là một đoạn liên tục `[cs, max(ce, e)]`. Nếu `s > ce`, không còn đoạn nào sau đó (điểm đầu `>= s`) chạm được tới đoạn gộp, nên ta chốt. Các đoạn gộp rời nhau nên cộng độ dài không trùng lặp.

## 11. Phân tích Time / Space Complexity

`n = len(nums)`, `M = 100` (toạ độ lớn nhất), `L` là độ dài trung bình của một đoạn (`<= M`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. `set` mọi điểm | `O(n·L)` ≤ `O(n·M)` | `O(M)` | oracle |
| 2. Mảng đánh dấu | `O(n·L)` | `O(M)` | gán lát cắt nhanh |
| 3. Mảng hiệu + quét | `O(n + M)` | `O(M)` | **code nộp**, mỗi đoạn `O(1)` |
| 4. Sắp xếp + gộp đoạn | `O(n log n)` | `O(1)` phụ | độc lập với toạ độ |

Với `n <= 100`, `M <= 100` mọi cách tức thì. Nếu toạ độ rất lớn (`10^9`), chỉ cách 4 (hoặc mảng hiệu nén toạ độ) khả thi.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: một xe một điểm; một xe cả đoạn; hai xe trùng hẳn; xe này nằm gọn trong xe kia; hai xe **kề nhau** (không chung điểm); hai xe **chạm** tại một điểm (chung đúng một điểm); toạ độ biên `1` và `100`.

In [23]:
impls = {
    "set": points_set,
    "marks": points_marks,
    "diff": points_diff,
    "merge": points_merge,
    "Solution": Solution().numberOfPoints,
}
cases = [
    ("ví dụ 1", [[3, 6], [1, 5], [4, 7]], 7),
    ("ví dụ 2", [[1, 3], [5, 8]], 7),
    ("một xe một điểm", [[5, 5]], 1),
    ("một xe cả đoạn", [[1, 100]], 100),
    ("hai xe trùng hẳn", [[2, 6], [2, 6]], 5),
    ("xe nằm gọn trong xe khác", [[1, 10], [3, 4]], 10),
    ("hai xe kề nhau không chồng", [[1, 3], [4, 6]], 6),
    ("hai xe chạm tại một điểm", [[1, 3], [3, 5]], 5),
    ("biên 1 và 100", [[1, 1], [100, 100]], 2),
    ("nhiều xe rời nhau", [[1, 2], [4, 5], [7, 8]], 6),
    ("thứ tự đầu vào ngược", [[10, 12], [1, 3]], 6),
]
for name, nums, exp in cases:
    for k, f in impls.items():
        got = f([list(x) for x in nums])
        assert got == exp, (name, k, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 7
PASS ví dụ 2 -> 7
PASS một xe một điểm -> 1
PASS một xe cả đoạn -> 100
PASS hai xe trùng hẳn -> 5
PASS xe nằm gọn trong xe khác -> 10
PASS hai xe kề nhau không chồng -> 6
PASS hai xe chạm tại một điểm -> 5
PASS biên 1 và 100 -> 2
PASS nhiều xe rời nhau -> 6
PASS thứ tự đầu vào ngược -> 6


Kiểm tay vài ca: `[[1,3],[3,5]]` phủ `1, 2, 3, 4, 5` = 5 điểm (điểm `3` chỉ đếm một lần) ✓. `[[1,3],[4,6]]` phủ `1..6` = 6 điểm ✓ (không chồng nhưng vẫn đủ 6 điểm khác nhau). `[[1,10],[3,4]]` = 10 vì xe thứ hai nằm gọn bên trong ✓.

### 12.2. So sánh các cách trên nhiều input ngẫu nhiên có seed

Input hợp lệ theo constraints: `1 <= n <= 100`, `1 <= start <= end <= 100`. Ta trộn đoạn ngắn / dài để có lúc dày đặc, lúc thưa.

In [24]:
import random
random.seed(2848)
for _ in range(2000):
    n = random.randint(1, 100)
    max_len = random.choice([1, 3, 10, 100])
    nums = []
    for _ in range(n):
        s = random.randint(1, 100)
        e = min(100, s + random.randint(0, max_len))
        nums.append([s, e])
    exp = points_set(nums)
    for k, f in impls.items():
        assert f([list(x) for x in nums]) == exp, (nums, k)
print("2000 ca ngẫu nhiên: mọi cách khớp brute force")

2000 ca ngẫu nhiên: mọi cách khớp brute force


### 12.3. Kiểm tính chất

- Đáp án `>=` độ dài của đoạn dài nhất và `<=` tổng các độ dài.
- Đáp án `<= 100`.
- Hoán vị thứ tự các xe không đổi đáp án; thêm một xe không làm đáp án giảm.
- Không phá mảng đầu vào.

In [25]:
random.seed(28480)
for _ in range(500):
    n = random.randint(1, 30)
    nums = []
    for _ in range(n):
        s = random.randint(1, 100)
        e = random.randint(s, 100)
        nums.append([s, e])
    copy = [x[:] for x in nums]
    r = Solution().numberOfPoints(nums)
    assert nums == copy
    assert max(e - s + 1 for s, e in nums) <= r <= min(100, sum(e - s + 1 for s, e in nums))
    shuffled = copy[:]
    random.shuffle(shuffled)
    assert Solution().numberOfPoints(shuffled) == r
    s = random.randint(1, 100)
    assert Solution().numberOfPoints(nums + [[s, random.randint(s, 100)]]) >= r
print("tính chất đúng")

tính chất đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `range(start, end)` (mất điểm cuối)

In [26]:
def sai_mat_diem_cuoi(nums):
    seen = set()
    for s, e in nums:
        for p in range(s, e):          # sai: thiếu e
            seen.add(p)
    return len(seen)

print("sai :", sai_mat_diem_cuoi([[3, 6], [1, 5], [4, 7]]))     # 6
print("đúng:", Solution().numberOfPoints([[3, 6], [1, 5], [4, 7]]))   # 7

sai : 6
đúng: 7


**Vì sao sai:** đề nói xe phủ **cả hai đầu**, nên điểm `end` cũng bị phủ. Dùng `range(start, end + 1)`.

### Lỗi 2: Cộng độ dài các đoạn (đếm trùng)

In [27]:
def sai_cong_do_dai(nums):
    return sum(e - s + 1 for s, e in nums)

print("sai :", sai_cong_do_dai([[3, 6], [1, 5], [4, 7]]))        # 13
print("đúng:", Solution().numberOfPoints([[3, 6], [1, 5], [4, 7]]))   # 7

sai : 13
đúng: 7


**Vì sao sai:** các đoạn **chồng nhau**; điểm được phủ bởi nhiều xe bị đếm nhiều lần. Cần `set`, đánh dấu, hoặc gộp đoạn.

### Lỗi 3: Mảng hiệu nhưng quên trừ tại `e + 1`

In [28]:
def sai_diff_thieu_tru(nums):
    diff = [0] * 102
    for s, e in nums:
        diff[s] += 1                    # sai: thiếu diff[e + 1] -= 1
    cur = count = 0
    for i in range(1, 101):
        cur += diff[i]
        if cur > 0:
            count += 1
    return count

print("sai :", sai_diff_thieu_tru([[1, 3], [5, 8]]))      # 100 (phủ tới hết 100)
print("đúng:", Solution().numberOfPoints([[1, 3], [5, 8]]))    # 7

sai : 100
đúng: 7


**Vì sao sai:** không có `-1` tại `e + 1` thì xe "phủ mãi mãi" từ `s` trở đi. Cần cả hai cập nhật: `+1` tại `s`, `−1` tại `e + 1`.

### Lỗi 4: Gộp đoạn mà không sắp xếp

In [29]:
def sai_gop_khong_sap_xep(nums):
    cs, ce = nums[0]
    total = 0
    for s, e in nums[1:]:
        if s <= ce:
            ce = max(ce, e)
        else:
            total += ce - cs + 1
            cs, ce = s, e
    return total + ce - cs + 1

print("sai :", sai_gop_khong_sap_xep([[3, 6], [1, 5], [4, 7]]))      # không sắp xếp
print("đúng:", Solution().numberOfPoints([[3, 6], [1, 5], [4, 7]]))   # 7

sai : 5
đúng: 7


**Vì sao sai:** thuật toán gộp đoạn giả định các đoạn đến theo thứ tự **điểm đầu tăng dần**. Phải `sorted(nums)` trước.

### Lỗi 5 (nhỏ): Mảng quá nhỏ làm `diff[e + 1]` vượt biên

Với `e = 100`, `e + 1 = 101` nên cần chỉ số `101`; mảng `[0] * 101` chỉ có chỉ số `0..100` ⇒ `IndexError`. Dùng `[0] * 102`.

In [30]:
try:
    diff = [0] * 101
    diff[100 + 1] -= 1
except IndexError as ex:
    print("IndexError:", ex)

IndexError: list index out of range


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "đếm số điểm / vị trí được phủ bởi ít nhất một đoạn" ⇒ **tập hợp**, **mảng đánh dấu**, **mảng hiệu**, hoặc **gộp đoạn**.
- Toạ độ nhỏ (`<= 100`) ⇒ đánh dấu là đơn giản nhất; toạ độ lớn ⇒ **gộp đoạn** (sắp xếp theo điểm đầu).
- Mẫu mảng hiệu: `diff[s] += 1; diff[e + 1] -= 1`, rồi cộng dồn; nhớ **kích thước `max + 2`**.
- Đoạn **đóng** `[s, e]` ⇒ `range(s, e + 1)`, độ dài `e − s + 1`.
- Điều kiện gộp đoạn cho điểm nguyên: `s <= ce` (chồng / chạm điểm); `s == ce + 1` là kề nhau nhưng không chung điểm.

## 15. Biến thể của bài

1. **Đếm điểm bị phủ bởi ít nhất `k` xe:** đổi điều kiện `cur > 0` thành `cur >= k` (mảng hiệu).
2. **Điểm bị phủ nhiều xe nhất:** `max` của `cur` (LeetCode 253 Meeting Rooms II dạng đếm tối đa).
3. **Toạ độ lớn (`10^9`):** gộp đoạn, hoặc mảng hiệu trên **toạ độ nén** (chỉ các `s`, `e + 1`).
4. **Số đoạn gộp rời nhau** (số "cụm"): đếm số lần "chốt đoạn" trong cách 4.
5. **Đoạn mở / nửa mở** (`[s, e)`): đổi điều kiện biên, `diff[e] -= 1`.
6. **2 chiều:** đếm ô được phủ bởi các hình chữ nhật (mảng hiệu 2D).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `set` mọi điểm | liệt kê điểm, `set` loại trùng | `O(n·M)` | `O(M)` | oracle, người mới |
| 2. Mảng đánh dấu | `covered[s:e+1] = [True]*...` | `O(n·M)` | `O(M)` | toạ độ nhỏ, code ngắn |
| 3. Mảng hiệu | `+1` tại `s`, `−1` tại `e + 1`, cộng dồn | `O(n + M)` | `O(M)` | **code nộp**, mở rộng cho "phủ ≥ k" |
| 4. Gộp đoạn | sắp xếp rồi gộp đoạn chồng | `O(n log n)` | `O(1)` phụ | toạ độ rất lớn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(range(2, 5 + 1))`, `len({1, 2, 2, 3})`, `sorted([[3, 6], [1, 5]])`, `max(5, 9)`.

**Bài 2 (tính tay).** Với `nums = [[2, 4], [6, 8], [3, 7]]`: vẽ các đoạn lên trục số, liệt kê điểm được phủ và đếm; sau đó chạy tay thuật toán gộp đoạn.

**Bài 3 (điền TODO).** Hoàn thành `points_diff_todo(nums)` theo Cách 3.

In [31]:
def points_diff_todo(nums):
    # TODO 1: diff = [0] * 102
    # TODO 2: với mỗi [s, e]: diff[s] += 1 và diff[e + 1] -= 1
    # TODO 3: cur = count = 0
    # TODO 4: với i từ 1 tới 100: cur += diff[i]; nếu cur > 0 thì count += 1
    # TODO 5: return count
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `points_merge2(nums)` bằng cách sắp xếp và gộp đoạn **không nhìn** code mẫu. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `points_at_least_k(nums, k)`: số điểm bị phủ bởi **ít nhất `k` xe**. (b) Viết `max_overlap(nums)`: **số xe tối đa** cùng phủ một điểm.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `range(2, 6)` cho `[2, 3, 4, 5]`; `{1, 2, 2, 3}` có 3 phần tử; `sorted` so sánh theo phần tử đầu. Bài 2: các điểm `2..8` đều được phủ (đoạn `[3, 7]` nối hai đoạn kia lại).
- **Mức 2 (vừa):** Bài 3: năm dòng đã ghi trong comment TODO. Bài 4: `sorted(nums)`, giữ `[cs, ce]`, nếu `s <= ce` thì `ce = max(ce, e)`, ngược lại chốt đoạn.
- **Mức 3 (sâu):** Bài 5(a): giống Cách 3 nhưng đếm `cur >= k`. Bài 5(b): trong lúc quét lưu `best = max(best, cur)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert list(range(2, 5 + 1)) == [2, 3, 4, 5]
assert len({1, 2, 2, 3}) == 3
assert sorted([[3, 6], [1, 5]]) == [[1, 5], [3, 6]]
assert max(5, 9) == 9

# Bài 2: [[2,4],[6,8],[3,7]]
# [3,7] nối [2,4] và [6,8] thành [2,8] -> 7 điểm
assert Solution().numberOfPoints([[2, 4], [6, 8], [3, 7]]) == 7

# Bài 3
def points_diff_todo_ans(nums):
    diff = [0] * 102
    for s, e in nums:
        diff[s] += 1
        diff[e + 1] -= 1
    cur = count = 0
    for i in range(1, 101):
        cur += diff[i]
        if cur > 0:
            count += 1
    return count

assert points_diff_todo_ans([[3, 6], [1, 5], [4, 7]]) == 7
assert points_diff_todo_ans([[1, 3], [5, 8]]) == 7

# Bài 4
def points_merge2(nums):
    nums = sorted(nums)
    cs, ce = nums[0]
    total = 0
    for s, e in nums[1:]:
        if s <= ce:
            ce = max(ce, e)
        else:
            total += ce - cs + 1
            cs, ce = s, e
    return total + ce - cs + 1

assert points_merge2([[3, 6], [1, 5], [4, 7]]) == 7
assert points_merge2([[1, 3], [5, 8]]) == 7

# Bài 5(a)
def points_at_least_k(nums, k):
    diff = [0] * 102
    for s, e in nums:
        diff[s] += 1
        diff[e + 1] -= 1
    cur = count = 0
    for i in range(1, 101):
        cur += diff[i]
        if cur >= k:
            count += 1
    return count

assert points_at_least_k([[3, 6], [1, 5], [4, 7]], 1) == 7
assert points_at_least_k([[3, 6], [1, 5], [4, 7]], 3) == 2      # điểm 4 và 5 có đủ 3 xe

# Bài 5(b)
def max_overlap(nums):
    diff = [0] * 102
    for s, e in nums:
        diff[s] += 1
        diff[e + 1] -= 1
    cur = best = 0
    for i in range(1, 101):
        cur += diff[i]
        best = max(best, cur)
    return best

assert max_overlap([[3, 6], [1, 5], [4, 7]]) == 3
assert max_overlap([[1, 3], [5, 8]]) == 1

random.seed(17)
for _ in range(300):
    nums = []
    for _ in range(random.randint(1, 20)):
        s = random.randint(1, 100)
        nums.append([s, random.randint(s, 100)])
    assert points_diff_todo_ans(nums) == points_merge2(nums) == points_set(nums)
    cover = [sum(1 for s, e in nums if s <= p <= e) for p in range(1, 101)]
    assert max_overlap(nums) == max(cover)
    for k in (1, 2, 3):
        assert points_at_least_k(nums, k) == sum(1 for c in cover if c >= k)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mảng hiệu: `+1` tại đầu, `−1` sau đuôi, cộng dồn)

```python
from typing import List

class Solution:
    def numberOfPoints(self, nums: List[List[int]]) -> int:
        diff = [0] * 102
        for s, e in nums:
            diff[s] += 1
            diff[e + 1] -= 1
        cur = count = 0
        for i in range(1, 101):
            cur += diff[i]
            if cur > 0:
                count += 1
        return count
```

**Checklist**

- [ ] Đoạn **đóng** hai đầu: dùng `e + 1` ở `range` / mảng hiệu?
- [ ] Không cộng độ dài các đoạn một cách ngây thơ (trùng lặp)?
- [ ] Mảng hiệu đủ kích thước (`102`) và có cả `+1` tại `s` lẫn `−1` tại `e + 1`?
- [ ] Nếu gộp đoạn: đã `sorted` theo điểm đầu và dùng `max` cho điểm cuối?
- [ ] Đã thử: một điểm, đoạn chồng, đoạn lồng nhau, kề nhau, chạm điểm, biên `1` / `100`?
- [ ] Đã so các cách với nhau trên input ngẫu nhiên có seed?
- [ ] Signature đúng `numberOfPoints(self, nums: List[List[int]]) -> int`?